In [ ]:
"""
MVTec Bottle Defect Segmentation — UNet (ResNet34 Encoder)
=============================================================
Production-grade semantic segmentation pipeline for pixel-wise defect
localization on MVTec AD bottle images.

Architecture:
    - Model     : UNet (segmentation_models_pytorch), ResNet34 encoder
    - Encoder   : ImageNet pretrained, fully fine-tuned (not frozen —
                  dense prediction benefits from low-level feature
                  adaptation more than classification does)
    - Loss      : Compound Loss — 0.5 x BCEWithLogits + 0.5 x Dice
    - Output    : Raw logits (activation=None) — numerically stable,
                  matches BCEWithLogitsLoss + DiceLoss(from_logits=True)

Dataset Engineering:
    Negative sampling — nominal ("good") images are paired with
    zero-tensor masks during training. Without this, the network
    would assume every image contains a defect somewhere, causing
    false-positive hallucinations in production. Validation set
    remains defect-only to measure performance on the hard cases
    that actually matter, while a small nominal subset is carved
    out separately for false-positive threshold calibration.

Deployment:
    FastAPI + Docker (CPU/GPU inference)

Author  : Alejandro Toro Arrabal
Version : 2.0.0
"""

# ── Standard Library
import copy
import logging
import os
import random
import sys
import tarfile
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from urllib.request import urlretrieve

# ── Third Party
import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

In [ ]:
# ════════════════════════════════════════════════════════
# 1. LOGGING INFRASTRUCTURE
# ════════════════════════════════════════════════════════

def setup_logger(name: str = "Factory_Segmentation") -> logging.Logger:
    """Configures a structured stdout logger with duplicate handler prevention."""
    logger = logging.getLogger(name)
    if logger.hasHandlers():
        logger.handlers.clear()
    logger.setLevel(logging.INFO)
    logger.propagate = False
    formatter = logging.Formatter(
        fmt="%(asctime)s - [%(levelname)s] - %(message)s",
        datefmt="%H:%M:%S"
    )
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    return logger


logger = setup_logger()

In [ ]:
# ════════════════════════════════════════════════════════
# 2. CONFIGURATION
# ════════════════════════════════════════════════════════

@dataclass(frozen=True)
class SegmentationConfig:
    """
    Immutable configuration for the UNet defect segmentation pipeline.

    DEFECT_AREA_THRESHOLD is a fallback default. The recommended
    production value is derived empirically via calibrate_threshold()
    against a nominal-only validation subset — see method docstring.
    """

    # ── Data Sources
    DATA_URL: str = "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420937370-1629958698/bottle.tar.xz"
    ARCHIVE_NAME: str = "bottle.tar.xz"
    DATA_ROOT: Path = Path("bottle")

    # ── Training Hyperparameters
    BATCH_SIZE: int = 16
    NUM_WORKERS: int = 2
    EPOCHS: int = 35
    LEARNING_RATE: float = 1e-4
    WEIGHT_DECAY: float = 1e-5
    SEED: int = 42

    # ── Checkpointing
    SAVE_PATH: str = "best_unet_bottle.pth"

    # ── Defect Detection
    # Fallback default — overridden by calibrate_threshold() in production.
    DEFECT_AREA_THRESHOLD: int = 50

    # ── Calibration
    NOMINAL_CALIBRATION_RATIO: float = 0.10  # 10% of good images held out for calibration

In [ ]:
# ════════════════════════════════════════════════════════
# 3. CUSTOM LOSS
# ════════════════════════════════════════════════════════

class CompoundLoss(nn.Module):
    """
    Combines BCEWithLogits (pixel-wise confidence) and Dice (spatial
    overlap, immune to class imbalance) for industrial defect segmentation.

    BCE alone fails on MVTec-scale class imbalance — defects often
    occupy <1% of the image, causing the network to predict all-zero
    masks with deceptively high pixel accuracy. Dice forces localization
    regardless of defect size; BCE provides stable gradients when Dice's
    intersection term is near zero early in training.
    """

    def __init__(self) -> None:
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss(mode="binary", from_logits=True)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        return 0.5 * self.bce(logits, targets) + 0.5 * self.dice(logits, targets)

In [ ]:
# ════════════════════════════════════════════════════════
# 4. DATA PIPELINE
# ════════════════════════════════════════════════════════

class MVTecSegmentationDataset(Dataset):
    """
    Yields (image, mask) pairs for supervised segmentation training.

    Nominal images receive a zero-tensor mask (mask_path=None) —
    negative sampling that prevents the network from hallucinating
    defects in production on perfectly normal images.
    """

    def __init__(
        self,
        image_paths: List[str],
        mask_paths: List[Optional[str]],
        transform: Optional[A.Compose] = None
    ) -> None:
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        image = cv2.imread(self.image_paths[idx])
        if image is None:
            raise ValueError(f"Corrupt or missing image: {self.image_paths[idx]}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask_path = self.mask_paths[idx]
        if mask_path is not None:
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask is None:
                raise ValueError(f"Corrupt or missing mask: {mask_path}")
            mask = np.where(mask > 127, 1.0, 0.0).astype(np.float32)
        else:
            # Negative sample — nominal image, zero-tensor mask
            mask = np.zeros(image.shape[:2], dtype=np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image, mask = augmented["image"], augmented["mask"]

        if mask.ndim == 2:
            mask = mask.unsqueeze(0)  # Albumentations ToTensorV2 doesn't add channel dim to masks

        return image, mask.float()

In [ ]:
# ════════════════════════════════════════════════════════
# 5. SEMANTIC AOI SYSTEM — OOP CONTROLLER
# ════════════════════════════════════════════════════════

class SemanticAOISystem:
    """
    End-to-end semantic segmentation pipeline for MVTec bottle defects.

    Responsibilities:
        - Dynamic dataset restructuring (auto-detects defect categories)
        - Negative-sample-aware DataLoader construction
        - UNet (ResNet34) training via compound BCE+Dice loss
        - IoU-primary checkpointing
        - Percentile-based defect threshold calibration on nominal data
        - Full test-set evaluation (IoU + Dice, per-category breakdown)
        - API-ready inference with morphological noise cleanup
        - Visual inspection grid: prediction overlay + IoU per sample

    Usage:
        config = SegmentationConfig(EPOCHS=35)
        system = SemanticAOISystem(config)
        system.ingest_data()
        system.prepare_dataloaders()
        system.train()
        system.calibrate_threshold()
        system.evaluate_test_set()
    """

    def __init__(self, config: SegmentationConfig) -> None:
        self.cfg = config
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        self.model: Optional[nn.Module] = None
        self.dataloaders: Dict = {}
        self.dataset_sizes: Dict = {}
        self.nominal_calibration_files: List[str] = []
        self.calibrated_threshold: Optional[int] = None

        self._set_seed()
        logger.info(f"Computation Device: {self.device.type.upper()}")

    # ────────────────────────────────────────────────────
    # PRIVATE UTILITIES
    # ────────────────────────────────────────────────────

    def _set_seed(self) -> None:
        """
        Locks all random number generators for full reproducibility.
        ~10-20% training speed penalty — re-enable benchmark=True for
        production inference where input sizes are fixed.
        """
        seed = self.cfg.SEED
        os.environ["PYTHONHASHSEED"] = str(seed)
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        if torch.cuda.is_available():
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
        logger.info(f"Deterministic seed locked: {seed}")

    @staticmethod
    def _build_transform(train: bool = False) -> A.Compose:
        """
        Shared preprocessing pipeline for train, val, calibration, and
        inference. CRITICAL: the val/inference variant must be used
        identically everywhere the model sees an image outside training —
        any inconsistency causes silently degraded predictions.
        """
        if train:
            return A.Compose([
                A.Resize(256, 256),
                A.HorizontalFlip(p=0.5),
                A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
                A.RandomBrightnessContrast(p=0.2),
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
                ToTensorV2(),
            ])
        return A.Compose([
            A.Resize(256, 256),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])

    @staticmethod
    def _calculate_iou(pred_mask: torch.Tensor, true_mask: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
        """Intersection over Union on binary masks."""
        intersection = (pred_mask & true_mask).sum().float()
        union = (pred_mask | true_mask).sum().float()
        return (intersection + eps) / (union + eps)

    @staticmethod
    def _calculate_dice(pred_mask: torch.Tensor, true_mask: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
        """Dice coefficient (F1 at pixel level) on binary masks."""
        intersection = (pred_mask & true_mask).sum().float()
        total = pred_mask.sum().float() + true_mask.sum().float()
        return (2 * intersection + eps) / (total + eps)

    def _get_defect_categories(self) -> List[str]:
        """
        Dynamically detects defect subfolder names under test/.
        Generalizes across any MVTec object category — no hardcoded
        defect names like 'broken_large' that only apply to bottles.
        """
        test_dir = self.cfg.DATA_ROOT / "test"
        return sorted([
            d.name for d in test_dir.iterdir()
            if d.is_dir() and d.name != "good"
        ])

    def _build_mask_path(self, image_path: Path) -> Path:
        """
        Constructs the ground truth mask path from an image path using
        robust pathlib manipulation — not fragile string replacement.

        MVTec structure: test/{defect}/{id}.png -> ground_truth/{defect}/{id}_mask.png
        """
        relative = image_path.relative_to(self.cfg.DATA_ROOT / "test")
        return (
            self.cfg.DATA_ROOT / "ground_truth" / relative.parent /
            f"{image_path.stem}_mask.png"
        )

    # ────────────────────────────────────────────────────
    # PUBLIC PIPELINE METHODS
    # ────────────────────────────────────────────────────

    def ingest_data(self) -> None:
        """Downloads and extracts the MVTec AD bottle dataset if not present."""
        if self.cfg.DATA_ROOT.exists():
            logger.info("Dataset found. Skipping download.")
            return

        try:
            logger.info("Downloading MVTec Bottle Dataset (~200MB)...")
            urlretrieve(self.cfg.DATA_URL, self.cfg.ARCHIVE_NAME)
            with tarfile.open(self.cfg.ARCHIVE_NAME, "r:xz") as tar:
                tar.extractall()
            os.remove(self.cfg.ARCHIVE_NAME)
            logger.info("Dataset extracted.")
        except Exception as e:
            logger.error(f"Dataset ingestion failed: {e}")
            raise RuntimeError(f"Failed to download/extract dataset: {e}") from e

    def prepare_dataloaders(self) -> None:
        """
        Constructs train/val DataLoaders with negative sampling and
        carves out a nominal-only calibration subset.

        Split strategy:
            - Defect images: 80% train, 20% val (val stays defect-only
              so IoU reflects performance on the hard cases)
            - Good images: 90% train (negative samples), 10% held out
              separately for false-positive threshold calibration
        """
        test_dir = self.cfg.DATA_ROOT / "test"
        train_good_dir = self.cfg.DATA_ROOT / "train" / "good"

        defect_categories = self._get_defect_categories()
        logger.info(f"Detected defect categories: {defect_categories}")

        defect_images: List[str] = []
        defect_masks: List[str] = []
        for defect in defect_categories:
            files = sorted((test_dir / defect).glob("*.png"))
            defect_images.extend([str(f) for f in files])
            defect_masks.extend([str(self._build_mask_path(f)) for f in files])

        train_def_img, val_def_img, train_def_mask, val_def_mask = train_test_split(
            defect_images, defect_masks, test_size=0.2, random_state=self.cfg.SEED
        )

        good_images = sorted([str(f) for f in train_good_dir.glob("*.png")])
        good_train, good_calib = train_test_split(
            good_images, test_size=self.cfg.NOMINAL_CALIBRATION_RATIO, random_state=self.cfg.SEED
        )
        self.nominal_calibration_files = good_calib

        train_files = train_def_img + good_train
        train_masks = train_def_mask + ([None] * len(good_train))
        val_files, val_masks = val_def_img, val_def_mask

        self.dataloaders = {
            "train": DataLoader(
                MVTecSegmentationDataset(train_files, train_masks, self._build_transform(train=True)),
                batch_size=self.cfg.BATCH_SIZE, shuffle=True,
                num_workers=self.cfg.NUM_WORKERS, pin_memory=True,
            ),
            "val": DataLoader(
                MVTecSegmentationDataset(val_files, val_masks, self._build_transform(train=False)),
                batch_size=self.cfg.BATCH_SIZE, shuffle=False,
                num_workers=self.cfg.NUM_WORKERS, pin_memory=True,
            ),
        }
        self.dataset_sizes = {"train": len(train_files), "val": len(val_files)}

        logger.info(
            f"Pipelines ready | Train: {len(train_files)} (incl. {len(good_train)} negative samples) | "
            f"Val: {len(val_files)} (defect-only) | Calibration: {len(good_calib)} (nominal-only)"
        )

    def build_model(self) -> None:
        """
        Initializes UNet with ResNet34 encoder, compound loss, AdamW.

        Encoder is NOT frozen — unlike classification transfer learning,
        dense pixel-wise prediction benefits from adapting low-level
        features to the specific defect textures, not just reusing
        generic ImageNet features as-is.
        """
        self.model = smp.Unet(
            encoder_name="resnet34",
            encoder_weights="imagenet",
            in_channels=3,
            classes=1,
            activation=None,  # raw logits — required for BCEWithLogitsLoss numerical stability
        ).to(self.device)

        self.criterion = CompoundLoss()
        self.optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=self.cfg.LEARNING_RATE,
            weight_decay=self.cfg.WEIGHT_DECAY,
        )
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode="min", factor=0.1, patience=3
        )
        logger.info("UNet (ResNet34 encoder) architecture initialized.")

    def train(self) -> None:
        """Executes training loop, checkpointing on best validation IoU."""
        if self.model is None:
            self.build_model()

        best_wts = copy.deepcopy(self.model.state_dict())
        best_iou = 0.0

        logger.info(f"Starting segmentation training ({self.cfg.EPOCHS} epochs)...")

        try:
            for epoch in range(self.cfg.EPOCHS):
                for phase in ["train", "val"]:
                    self.model.train() if phase == "train" else self.model.eval()
                    running_loss, running_iou = 0.0, 0.0

                    pbar = tqdm(
                        self.dataloaders[phase],
                        desc=f"Epoch {epoch + 1} [{phase.upper()}]",
                        leave=False,
                    )
                    for inputs, masks in pbar:
                        inputs, masks = inputs.to(self.device), masks.to(self.device)
                        self.optimizer.zero_grad()

                        with torch.set_grad_enabled(phase == "train"):
                            outputs = self.model(inputs)
                            loss = self.criterion(outputs, masks)

                            # Thresholding logits at 0 == thresholding sigmoid at 0.5
                            # Skips redundant sigmoid computation during metric calculation
                            preds = (outputs > 0).byte()
                            true_masks = (masks > 0.5).byte()

                            if phase == "train":
                                loss.backward()
                                self.optimizer.step()

                        batch_size = inputs.size(0)
                        running_loss += loss.item() * batch_size
                        running_iou += self._calculate_iou(preds, true_masks).item() * batch_size
                        pbar.set_postfix(loss=loss.item())

                    epoch_loss = running_loss / self.dataset_sizes[phase]
                    epoch_iou = running_iou / self.dataset_sizes[phase]

                    if phase == "val":
                        logger.info(
                            f"Epoch {epoch + 1}/{self.cfg.EPOCHS} | "
                            f"VAL Loss: {epoch_loss:.4f} | VAL IoU: {epoch_iou:.4f}"
                        )
                        self.scheduler.step(epoch_loss)

                        if epoch_iou > best_iou:
                            best_iou = epoch_iou
                            best_wts = copy.deepcopy(self.model.state_dict())
                            torch.save(self.model.state_dict(), self.cfg.SAVE_PATH)
                            logger.info(f"New best model saved (IoU: {best_iou:.4f})")

        except Exception as e:
            logger.error(f"Training failed: {e}")
            raise RuntimeError(f"Training aborted: {e}") from e

        self.model.load_state_dict(best_wts)
        logger.info(f"Training complete. Best validation IoU: {best_iou:.4f}")

    def load_model(self, path: str) -> None:
        """Loads model weights and runs warmup inference."""
        if not os.path.exists(path):
            raise FileNotFoundError(f"Model artifact not found: {path}")

        if self.model is None:
            self.build_model()

        self.model.load_state_dict(torch.load(path, map_location=self.device))
        logger.info(f"Weights loaded from: {path}")

        logger.info("Running warmup inference...")
        dummy = torch.zeros(1, 3, 256, 256).to(self.device)
        with torch.inference_mode():
            self.model(dummy)
        logger.info("Model warmed up and ready.")

    # ────────────────────────────────────────────────────
    # CALIBRATION & EVALUATION
    # ────────────────────────────────────────────────────

    def calibrate_threshold(self, percentile: float = 99.0) -> int:
        """
        Derives DEFECT_AREA_THRESHOLD empirically from nominal-only data —
        replaces the hardcoded magic number default.

        Methodology:
            Run inference on the held-out nominal calibration set (images
            the model never saw labeled defects for). These should predict
            near-zero defect pixels. Set the threshold at the Nth percentile
            of predicted pixel counts — controlling the false positive rate
            on known-good images to (100-N)%.

        Mirrors the percentile-based calibration approach used in the
        unsupervised anomaly detector for methodological consistency
        across the portfolio.
        """
        if self.model is None:
            raise RuntimeError("Model not loaded. Call load_model() or train() first.")
        if not self.nominal_calibration_files:
            raise RuntimeError("No calibration files available. Call prepare_dataloaders() first.")

        logger.info(f"Calibrating defect threshold at {percentile}th percentile...")
        self.model.eval()
        pixel_counts: List[int] = []

        for img_path in self.nominal_calibration_files:
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            result = self.predict_segmentation(img, apply_threshold=False)
            pixel_counts.append(result["defect_pixel_count"])

        threshold = int(np.percentile(pixel_counts, percentile))
        self.calibrated_threshold = threshold

        logger.info(
            f"Calibrated threshold: {threshold} pixels "
            f"(based on {len(pixel_counts)} nominal calibration images)"
        )
        return threshold

    def evaluate_test_set(self) -> Dict[str, Any]:
        """
        Comprehensive evaluation across the full labeled test set.

        Reports overall and per-category IoU and Dice — the standard
        segmentation benchmark metrics. Pixel accuracy is deliberately
        excluded as a primary metric since it is misleading on the
        extreme class imbalance typical of industrial defects.
        """
        if self.model is None:
            raise RuntimeError("Model not loaded.")

        self.model.eval()
        transform = self._build_transform(train=False)
        defect_categories = self._get_defect_categories()

        results: Dict[str, Any] = {"per_category": {}, "overall_iou": 0.0, "overall_dice": 0.0}
        all_ious, all_dices = [], []

        logger.info("Evaluating full test set...")
        for category in defect_categories:
            cat_dir = self.cfg.DATA_ROOT / "test" / category
            ious, dices = [], []

            for img_path in sorted(cat_dir.glob("*.png")):
                img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
                mask_path = self._build_mask_path(img_path)
                true_mask_raw = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
                true_mask = np.where(true_mask_raw > 127, 1, 0).astype(np.uint8)

                tensor = transform(image=img)["image"].unsqueeze(0).to(self.device)
                with torch.inference_mode():
                    logits = self.model(tensor)
                    pred_mask = (logits > 0).squeeze().byte().cpu()

                true_mask_resized = cv2.resize(
                    true_mask, (pred_mask.shape[1], pred_mask.shape[0]),
                    interpolation=cv2.INTER_NEAREST
                )
                true_tensor = torch.from_numpy(true_mask_resized).byte()

                ious.append(self._calculate_iou(pred_mask, true_tensor).item())
                dices.append(self._calculate_dice(pred_mask, true_tensor).item())

            results["per_category"][category] = {
                "mean_iou": float(np.mean(ious)),
                "mean_dice": float(np.mean(dices)),
                "samples": len(ious),
            }
            all_ious.extend(ious)
            all_dices.extend(dices)
            logger.info(
                f"  {category}: IoU={np.mean(ious):.4f} | Dice={np.mean(dices):.4f} | n={len(ious)}"
            )

        results["overall_iou"] = float(np.mean(all_ious))
        results["overall_dice"] = float(np.mean(all_dices))
        logger.info(
            f"Overall | IoU: {results['overall_iou']:.4f} | Dice: {results['overall_dice']:.4f}"
        )
        return results

    def predict_segmentation(
        self, image: np.ndarray, apply_threshold: bool = True
    ) -> Dict[str, Any]:
        """
        API-ready inference method. Accepts numpy arrays from FastAPI's
        in-memory byte decoding pipeline.

        Args:
            image: RGB numpy array of shape (H, W, 3), dtype uint8.
                   Note: OpenCV reads BGR — convert before passing.
            apply_threshold: If False, skips the is_defect decision —
                   used internally during calibration to gather raw
                   pixel counts before a threshold exists.

        Returns:
            JSON-serializable dict:
            {
                "is_defect": bool,
                "defect_pixel_count": int,
                "threshold": int
            }

        Pipeline:
            1. Preprocess (resize, normalize) — identical to val transform
            2. Forward pass → raw logits
            3. Sigmoid + threshold at 0.5 → binary mask
            4. Morphological opening — removes small noisy false-positive
               speckle before counting (the segmentation equivalent of
               Gaussian blur smoothing on continuous anomaly maps)
            5. Count defect pixels, compare against calibrated threshold
        """
        if image is None or image.size == 0:
            raise ValueError("Invalid image — array is None or empty.")
        if self.model is None:
            raise RuntimeError("Model not loaded. Call load_model() before predict_segmentation().")

        transform = self._build_transform(train=False)
        input_tensor = transform(image=image)["image"].unsqueeze(0).to(self.device)

        self.model.eval()
        with torch.inference_mode():
            logits = self.model(input_tensor)
            probs = torch.sigmoid(logits)
            pred_mask = (probs > 0.5).squeeze().cpu().numpy().astype(np.uint8)

        # Morphological opening — erosion then dilation, removes small
        # noisy blobs while preserving genuine structural defects
        kernel = np.ones((3, 3), np.uint8)
        pred_mask_clean = cv2.morphologyEx(pred_mask, cv2.MORPH_OPEN, kernel)

        defect_pixels = int(np.sum(pred_mask_clean))
        threshold = self.calibrated_threshold or self.cfg.DEFECT_AREA_THRESHOLD

        result = {
            "defect_pixel_count": defect_pixels,
            "threshold": threshold,
        }
        if apply_threshold:
            result["is_defect"] = defect_pixels > threshold

        return result

    def smoke_test(self) -> bool:
        """
        Validates the inference pipeline before deployment.

        Tests against a known defective image — verifies the pipeline
        runs end-to-end and produces a structurally valid response.
        """
        if self.model is None:
            logger.error("Smoke test failed: model not loaded.")
            return False

        try:
            defect_categories = self._get_defect_categories()
            if not defect_categories:
                logger.error("Smoke test failed: no defect categories found.")
                return False

            test_files = list((self.cfg.DATA_ROOT / "test" / defect_categories[0]).glob("*.png"))
            if not test_files:
                logger.error("Smoke test failed: no test images found.")
                return False

            img = cv2.cvtColor(cv2.imread(str(test_files[0])), cv2.COLOR_BGR2RGB)
            result = self.predict_segmentation(img)

            assert isinstance(result["is_defect"], bool), "is_defect must be bool"
            assert isinstance(result["defect_pixel_count"], int), "defect_pixel_count must be int"
            assert result["defect_pixel_count"] >= 0, "pixel count cannot be negative"

            logger.info(f"Smoke test passed | Result: {result}")
            return True

        except Exception as e:
            logger.error(f"Smoke test failed: {e}")
            return False

    def visualize_predictions(self, num_images: int = 4, save_path: str = "segmentation_grid.png") -> None:
        """QA tool — overlays predicted mask on input image with per-sample IoU."""
        if self.model is None:
            logger.error("Model not loaded. Cannot visualize.")
            return

        logger.info("Generating semantic overlays...")
        self.model.eval()

        fig, axs = plt.subplots(num_images // 2, 2, figsize=(10, 5 * (num_images // 2)))
        axs = axs.flatten()

        with torch.inference_mode():
            inputs, masks = next(iter(self.dataloaders["val"]))
            inputs, masks = inputs.to(self.device), masks.to(self.device)

            logits = self.model(inputs)
            preds = (torch.sigmoid(logits) > 0.5).float()

            for j in range(min(num_images, inputs.size(0))):
                img_np = np.clip(
                    inputs[j].cpu().permute(1, 2, 0).numpy() * np.array([0.229, 0.224, 0.225])
                    + np.array([0.485, 0.456, 0.406]), 0, 1
                )
                pred_mask = preds[j].cpu().squeeze().numpy()
                iou = self._calculate_iou(preds[j].byte(), masks[j].byte()).item()

                axs[j].imshow(img_np)
                axs[j].imshow(pred_mask, alpha=0.5, cmap="Reds")  # Red = prediction overlay
                axs[j].set_title(f"IoU: {iou:.2f}", fontweight="bold")
                axs[j].axis("off")

        plt.tight_layout()
        plt.savefig(save_path)
        logger.info(f"Overlay grid saved to: {save_path}")
        plt.show()

In [ ]:
# ════════════════════════════════════════════════════════
# 6. EXECUTION ENTRYPOINT
# ════════════════════════════════════════════════════════

if __name__ == "__main__":

    config = SegmentationConfig(EPOCHS=35)
    system = SemanticAOISystem(config)

    # ── Phase 1: Training pipeline
    system.ingest_data()
    system.prepare_dataloaders()
    system.train()

    # ── Phase 2: Threshold calibration (replaces hardcoded magic number)
    system.calibrate_threshold(percentile=99.0)

    # ── Phase 3: Full test-set benchmark — IoU + Dice, per category
    system.evaluate_test_set()

    # ── Phase 4: Production validation
    prod_system = SemanticAOISystem(config)
    prod_system.load_model(config.SAVE_PATH)
    prod_system.calibrated_threshold = system.calibrated_threshold

    if prod_system.smoke_test():
        logger.info("[SYSTEM SECURED] Artifact validated. Ready for Docker integration.")
        prod_system.prepare_dataloaders()
        prod_system.visualize_predictions(num_images=4)
    else:
        logger.error("[SYSTEM FAILURE] Smoke test failed. Do not deploy.")